In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from scipy.optimize import least_squares

from inverse_pinn_model import InversePINN

In [ ]:
# Experiment settings
# Keep the same overall design, but make all random pieces reproducible.
device = torch.device("cpu")
dtype = torch.float64

TRAIN_POOL_SIZE = 200

data_sizes = [
    2,
    30,
    50,
    100,
    120,
    150,
    200,
]

N_f = 2000
N_terminal = 500
N_boundary = 500

epsilon = 1e-4

hidden_width = 64
hidden_layers = 4

r_initial = 0.10
sigma_initial = 0.40

learning_rate = 1e-3
num_epochs = 30000

print_every = 5000
history_every = 100

# Real-data lambda setting selected from Notebook 11.1
lambda_data = 1.0
lambda_pde = 1.0
lambda_terminal = 0.01
lambda_boundary = 0.001

# Separate seeds so changing one random component does not change the others
SPLIT_SEED = 11
PHYSICS_SEED = 22
MODEL_SEED = 33

print("device =", device)
print("data_sizes =", data_sizes)
print()
print("N_f =", N_f)
print("N_terminal =", N_terminal)
print("N_boundary =", N_boundary)
print()
print("lambdas =", {
    "data": lambda_data,
    "pde": lambda_pde,
    "terminal": lambda_terminal,
    "boundary": lambda_boundary,
})
print()
print("seeds =", {
    "split": SPLIT_SEED,
    "physics": PHYSICS_SEED,
    "model": MODEL_SEED,
})


In [ ]:

# ------------------------------------------------------------
# Automatically locate NDX_call_data.csv
# ------------------------------------------------------------

from pathlib import Path

_filename = "NDX_call_data.csv"

_search_roots = [
    Path.cwd().resolve()
]

try:
    _notebook_dir = Path(__file__).resolve().parent
    _search_roots.append(_notebook_dir)
except NameError:
    pass


_candidates = []

for _root in _search_roots:

    _candidates.extend([
        _root / _filename,
        _root / "data" / _filename,
        _root.parent / _filename,
        _root.parent / "data" / _filename,
    ])


data_path = None


# First check common locations
for _candidate in _candidates:

    if _candidate.is_file():

        data_path = _candidate.resolve()
        break


# Then recursively search the current project
if data_path is None:

    for _root in _search_roots:

        try:

            _matches = list(
                _root.rglob(
                    _filename
                )
            )

        except Exception:

            _matches = []


        if _matches:

            data_path = (
                _matches[0]
                .resolve()
            )

            break


if data_path is None:

    raise FileNotFoundError(
        "\nCould not find NDX_call_data.csv.\n"
        "Make sure the CSV is included somewhere inside the downloaded project folder."
    )


print(
    "Using data file:",
    data_path
)


raw_df = pd.read_csv(
    data_path
)

unnamed_columns = [
    col
    for col in raw_df.columns
    if str(col).startswith("Unnamed:")
]

if unnamed_columns:
    raw_df = raw_df.drop(
        columns=unnamed_columns
    )


raw_df = raw_df.reset_index(
    drop=True
)


# Save original row identity
raw_df["original_row"] = np.arange(
    len(raw_df)
)


# If market_price is not already: use bid/ask midpoint
if "market_price" not in raw_df.columns:

    if not {
        "best_bid",
        "best_offer"
    }.issubset(
        raw_df.columns
    ):

        raise KeyError(
            "Need market_price or "
            "best_bid and best_offer."
        )

    raw_df["market_price"] = (
        raw_df["best_bid"]
        + raw_df["best_offer"]
    ) / 2.0

# Strike Price

K = float(
    raw_df[
        "strike_price"
    ]
    .dropna()
    .iloc[0]
)

# Keep required columns 
real_df = raw_df[
    [
        "original_row",
        "current_time",
        "spot_price",
        "market_price"
    ]
].copy()

# remove invalid observations
real_df = (
    real_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


real_df = real_df[
    (
        real_df["current_time"]
        >= 0.0
    )
    &
    (
        real_df["spot_price"]
        > 0.0
    )
    &
    (
        real_df["market_price"]
        >= 0.0
    )
].copy()


real_df = real_df.reset_index(
    drop=True
)


print(
    "Number of usable observations =",
    len(real_df)
)

print(
    "K =",
    K
)

In [ ]:
T = float(
    real_df[
        "current_time"
    ].max()
)


real_df["t"] = (
    T
    - real_df["current_time"]
)


# Physical stock price
real_df["S"] = (
    real_df["spot_price"]
)


# Physical option price
real_df["V"] = (
    real_df["market_price"]
)


print("T =", T)

print()

print(
    "t min =",
    real_df["t"].min()
)

print(
    "t max =",
    real_df["t"].max()
)

print()

print(
    "S min =",
    real_df["S"].min()
)

print(
    "S max =",
    real_df["S"].max()
)

print()

print(
    "V min =",
    real_df["V"].min()
)

print(
    "V max =",
    real_df["V"].max()
)


real_df.head()

In [ ]:
# Define physical domain
t_min = 0.0
t_max = T


S_min = 0.0


S_data_max = float(
    real_df[
        "S"
    ].max()
)

S_max = (
    10.0 * S_data_max
)


In [ ]:
# Convert real observations to tensor
t_all = torch.tensor(
    real_df[
        "t"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


S_all = torch.tensor(
    real_df[
        "S"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


V_all = torch.tensor(
    real_df[
        "V"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)

# u_all for PINN and V_all for regression
u_all = (
    V_all / K
)


N_obs = len(
    real_df
)


print(
    "N observations =",
    N_obs
)

print()

print(
    "t_all shape =",
    t_all.shape
)

print(
    "S_all shape =",
    S_all.shape
)

print(
    "V_all shape =",
    V_all.shape
)

print(
    "u_all shape =",
    u_all.shape
)

### Fixed train/test split and nested n_data subsets
- Randomly select the 200-point training pool once using `SPLIT_SEED`.
- The remaining 39 observations form one fixed test set.
- The n_data subsets are nested: n=2 is contained in n=30, ..., and n=200 is the full training pool.
- For each n_data, Regression and PINN use exactly the same observed training points.


In [ ]:
# Fixed random train/test split
split_rng = np.random.default_rng(
    SPLIT_SEED
)

permutation = split_rng.permutation(
    N_obs
)

# 200-point training pool and fixed 39-point test set
train_pool_idx_np = permutation[
    :TRAIN_POOL_SIZE
]

test_idx_np = permutation[
    TRAIN_POOL_SIZE:
]

# Convert to torch indices
train_pool_idx = torch.tensor(
    train_pool_idx_np,
    dtype=torch.long,
    device=device
)

test_idx = torch.tensor(
    test_idx_np,
    dtype=torch.long,
    device=device
)

# Nested n_data subsets.
# BOTH Regression and PINN will use sample_indices[n_data].
sample_indices = {
    n_data: train_pool_idx[:n_data]
    for n_data in data_sizes
}

# Fixed test set for every n_data and both methods
t_test = t_all[test_idx]
S_test = S_all[test_idx]
V_test = V_all[test_idx]
u_test = u_all[test_idx]


In [ ]:
# Helper: return the exact same observed subset for Regression and PINN
def get_training_subset(n_data):

    selected_idx = sample_indices[n_data]

    t_train = t_all[selected_idx]
    S_train = S_all[selected_idx]
    V_train = V_all[selected_idx]
    u_train = u_all[selected_idx]

    return (
        selected_idx,
        t_train,
        S_train,
        V_train,
        u_train,
    )


In [ ]:
# Safety checks

# n_data = 200 is exactly the full 200-point training pool
assert torch.equal(
    sample_indices[200],
    train_pool_idx
)

# Check nested datasets
for n_small, n_large in zip(
    data_sizes[:-1],
    data_sizes[1:],
):

    small_set = set(
        sample_indices[n_small]
        .cpu()
        .numpy()
        .tolist()
    )

    large_set = set(
        sample_indices[n_large]
        .cpu()
        .numpy()
        .tolist()
    )

    assert small_set.issubset(
        large_set
    )

# Check no overlap between train pool and test set
assert set(train_pool_idx_np).isdisjoint(
    set(test_idx_np)
)

print("Total observations =", N_obs)
print("Training pool =", len(train_pool_idx))
print("Test set =", len(test_idx))
print("Nested n_data subsets = OK")
print("Regression/PINN shared observed subsets = OK")


In [ ]:
# save exact realization of the train/test split
split_df = real_df[
    [
        "original_row",
        "t",
        "S",
        "V"
    ]
].copy()


split_df[
    "split"
] = "test"


split_df.loc[
    train_pool_idx_np,
    "split"
] = "train_pool"

# position inside the randomly ordered 200-point pool
train_order = np.full(
    N_obs,
    np.nan
)


for order, idx in enumerate(
    train_pool_idx_np,
    start=1
):

    train_order[
        idx
    ] = order


split_df[
    "train_pool_order"
] = train_order


split_df.to_csv(
    "notebook11_train_test_split.csv",
    index=False
)



### Black-Scholes Eq.(5)

In [ ]:
def black_scholes_call(
    t,
    S,
    K,
    T,
    r,
    sigma
):

    tau = (
        T - t
    )


    eps = 1e-12


    tau_safe = torch.clamp(
        tau,
        min=eps
    )


    S_safe = torch.clamp(
        S,
        min=eps
    )

# d1
    d1 = (

        torch.log(
            S_safe / K
        )

        + (
            r
            + 0.5
            * sigma ** 2
        )
        * tau_safe

    ) / (

        sigma
        * torch.sqrt(
            tau_safe
        )
    )

# d2
    d2 = (

        d1

        - sigma
        * torch.sqrt(
            tau_safe
        )
    )


    # Normal CDF
    N_d1 = (

        0.5

        * (
            1.0

            + torch.erf(
                d1
                / np.sqrt(2.0)
            )
        )
    )


    N_d2 = (

        0.5

        * (
            1.0

            + torch.erf(
                d2
                / np.sqrt(2.0)
            )
        )
    )

  # BS call price
    V = (

        S_safe
        * N_d1

        - K
        * torch.exp(
            -r
            * tau_safe
        )
        * N_d2
    )


    # Exact terminal payoff
    payoff = torch.relu(
        S - K
    )


    V = torch.where(
        tau <= eps,
        payoff,
        V
    )
    # payoff =  max(S-K, 0)

    V = torch.where(
        S <= eps,
        torch.zeros_like(V),
        V
    )


    return V


### Regression calibration for each n_data
For every n_data, Regression is re-fit using the **same `sample_indices[n_data]` observations** used by the PINN. Therefore `r_tilde`, `sigma_tilde`, Regression train MSE, and Regression test MSE are allowed to change with n_data.


In [ ]:
def fit_regression_for_n_data(
    n_data
):

    (
        selected_idx,
        t_reg_train,
        S_reg_train,
        V_reg_train,
        _
    ) = get_training_subset(
        n_data
    )

    # residual = BS price - real market price
    def regression_residuals(
        parameters
    ):

        r_value = float(
            parameters[0]
        )

        sigma_value = float(
            parameters[1]
        )

        with torch.no_grad():

            V_prediction = black_scholes_call(
                t=t_reg_train,
                S=S_reg_train,
                K=K,
                T=T,
                r=r_value,
                sigma=sigma_value
            )

        residuals = (
            V_prediction
            - V_reg_train
        )

        return (
            residuals
            .cpu()
            .numpy()
            .reshape(-1)
        )

    # Nonlinear least-squares calibration
    regression_fit = least_squares(
        regression_residuals,
        x0=np.array(
            [
                r_initial,
                sigma_initial
            ],
            dtype=float
        ),
        bounds=(
            np.array(
                [
                    1e-8,
                    1e-8
                ]
            ),
            np.array(
                [
                    np.inf,
                    np.inf
                ]
            )
        ),
        max_nfev=10000
    )

    r_tilde = float(
        regression_fit.x[0]
    )

    sigma_tilde = float(
        regression_fit.x[1]
    )

    with torch.no_grad():

        # Training prediction for this n_data subset
        V_reg_train_prediction = black_scholes_call(
            t=t_reg_train,
            S=S_reg_train,
            K=K,
            T=T,
            r=r_tilde,
            sigma=sigma_tilde
        )

        regression_train_mse = torch.mean(
            (
                V_reg_train_prediction
                - V_reg_train
            ) ** 2
        ).item()

        # Same fixed test set for every n_data
        V_regression_test = black_scholes_call(
            t=t_test,
            S=S_test,
            K=K,
            T=T,
            r=r_tilde,
            sigma=sigma_tilde
        )

        mse_1_regression = torch.mean(
            (
                V_regression_test
                - V_test
            ) ** 2
        ).item()

    result = {
        "n_data": n_data,
        "r_tilde": r_tilde,
        "sigma_tilde": sigma_tilde,
        "regression_train_mse": regression_train_mse,
        "mse_1_regression_eq5_test": mse_1_regression,
        "regression_success": bool(regression_fit.success),
    }

    print()
    print("-" * 70)
    print(f"Regression with n_data = {n_data}")
    print(f"r_tilde = {r_tilde:.10f}")
    print(f"sigma_tilde = {sigma_tilde:.10f}")
    print(f"Regression train MSE = {regression_train_mse:.10f}")
    print(f"Regression test MSE = {mse_1_regression:.10f}")
    print("Optimization success =", regression_fit.success)

    return (
        result,
        V_regression_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


### MSE #1: Regression
`MSE #1` is now computed separately inside `fit_regression_for_n_data(n_data)` for every n_data value, always on the same fixed test set.


In [ ]:
# No one-time 200-point Regression fit here anymore.
# Regression is re-fit inside the main n_data loop so its parameters
# and test MSE can change with n_data.


In [ ]:
# BS PDE residual

def black_scholes_residual(
    model,
    t,
    S
):

    t_input = (
        t.clone()
        .detach()
        .requires_grad_(True)
    )


    S_input = (
        S.clone()
        .detach()
        .requires_grad_(True)
    )


    u = model(
        t_input,
        S_input
    )


    # du/dt
    u_t = torch.autograd.grad(

        outputs=u,
        inputs=t_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # du/dS
    u_S = torch.autograd.grad(

        outputs=u,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # d2u/dS2
    u_SS = torch.autograd.grad(

        outputs=u_S,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u_S
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    residual = (

        u_t

        + 0.5
        * model.sigma ** 2
        * S_input ** 2
        * u_SS

        + model.r
        * S_input
        * u_S

        - model.r
        * u
    )


    return residual


### PINN PDE residual + loss

In [ ]:
# PINN loss
def compute_losses(

    model,

    t_train,
    S_train,
    u_train,

    t_f,
    S_f,

    t_terminal,
    S_terminal,

    t_boundary,
    S_boundary_left,
    S_boundary_right
):

    # 1. Data loss

    u_data_prediction = model(
        t_train,
        S_train
    )


    loss_data = torch.mean(

        (
            u_data_prediction
            - u_train
        ) ** 2
    )


    # 2. PDE loss

    pde_prediction = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )


    loss_pde = torch.mean(
        pde_prediction ** 2
    )


    # 3. Terminal loss
    #
    # V(T,S) = max(S-K,0)
    #
    # u(T,S) = V/K

    u_terminal_prediction = model(
        t_terminal,
        S_terminal
    )


    u_terminal_true = (

        torch.relu(
            S_terminal
            - K
        )

        / K
    )


    loss_terminal = torch.mean(

        (
            u_terminal_prediction
            - u_terminal_true
        ) ** 2
    )

    # 4. Left boundary
    #
    # V(t,0) = 0

    u_left_prediction = model(
        t_boundary,
        S_boundary_left
    )


    u_left_true = (
        torch.zeros_like(
            u_left_prediction
        )
    )


    loss_boundary_left = torch.mean(

        (
            u_left_prediction
            - u_left_true
        ) ** 2
    )


    # 5. Right boundary
    #
    # V approximately
    #
    # S - K exp[-r(T-t)]

    u_right_prediction = model(
        t_boundary,
        S_boundary_right
    )


    u_right_true = (

        S_boundary_right

        - K
        * torch.exp(

            -model.r

            * (
                T
                - t_boundary
            )
        )

    ) / K


    loss_boundary_right = torch.mean(

        (
            u_right_prediction
            - u_right_true
        ) ** 2
    )


    loss_boundary = (

        loss_boundary_left
        + loss_boundary_right
    )


    # Total loss

    total_loss = (

        lambda_data
        * loss_data

        + lambda_pde
        * loss_pde

        + lambda_terminal
        * loss_terminal

        + lambda_boundary
        * loss_boundary
    )


    return {

        "total":
            total_loss,

        "data":
            loss_data,

        "pde":
            loss_pde,

        "terminal":
            loss_terminal,

        "boundary":
            loss_boundary
    }

In [ ]:
# Fixed shared PINN physics points
# Regression does NOT use collocation / terminal / boundary points.
# All seven PINNs use exactly the same fixed physics points.

torch.manual_seed(
    PHYSICS_SEED
)

# PDE collocation points
# t in (t_min, t_max), S in (S_min, S_max)
t_f = (
    t_min
    + epsilon
    + (
        t_max
        - t_min
        - 2.0 * epsilon
    )
    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

S_f = (
    S_min
    + epsilon
    + (
        S_max
        - S_min
        - 2.0 * epsilon
    )
    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

# Terminal-condition points
# t = T, S in [S_min, S_max]
t_terminal = torch.full(
    (N_terminal, 1),
    fill_value=T,
    dtype=dtype,
    device=device
)

S_terminal = (
    S_min
    + (
        S_max
        - S_min
    )
    * torch.rand(
        N_terminal,
        1,
        dtype=dtype,
        device=device
    )
)

# Boundary-condition points
# same t_boundary is used for left and right boundary
t_boundary = (
    t_min
    + (
        t_max
        - t_min
    )
    * torch.rand(
        N_boundary,
        1,
        dtype=dtype,
        device=device
    )
)

S_boundary_left = torch.full(
    (N_boundary, 1),
    fill_value=S_min,
    dtype=dtype,
    device=device
)

S_boundary_right = torch.full(
    (N_boundary, 1),
    fill_value=S_max,
    dtype=dtype,
    device=device
)

print("Fixed shared physics points created")
print("PHYSICS_SEED =", PHYSICS_SEED)
print()
print("PDE t range:", float(t_f.min()), "to", float(t_f.max()))
print("PDE S range:", float(S_f.min()), "to", float(S_f.max()))
print("Terminal S range:", float(S_terminal.min()), "to", float(S_terminal.max()))
print("Right boundary S =", float(S_boundary_right[0]))


In [ ]:
# Create ONE fixed random initialization.
# Every n_data PINN starts from exactly this same state.
torch.manual_seed(
    MODEL_SEED
)

base_model = InversePINN(
    t_min=t_min,
    t_max=t_max,
    S_min=S_min,
    S_max=S_max,
    hidden_width=hidden_width,
    hidden_layers=hidden_layers,
    r_initial=r_initial,
    sigma_initial=sigma_initial
).to(
    device=device,
    dtype=dtype
)

base_state_dict = copy.deepcopy(
    base_model.state_dict()
)

print("MODEL_SEED =", MODEL_SEED)
print("Initial r =", base_model.r.item())
print("Initial sigma =", base_model.sigma.item())

del base_model


In [ ]:
# Save the exact fixed realization used by Notebook 11
# No new random points are generated in this cell.
torch.save(
    {
        "split_seed": SPLIT_SEED,
        "physics_seed": PHYSICS_SEED,
        "model_seed": MODEL_SEED,
        "train_pool_indices": train_pool_idx.cpu(),
        "test_indices": test_idx.cpu(),
        "sample_indices": {
            int(n): sample_indices[n].cpu()
            for n in data_sizes
        },
        "t_f": t_f.cpu(),
        "S_f": S_f.cpu(),
        "t_terminal": t_terminal.cpu(),
        "S_terminal": S_terminal.cpu(),
        "t_boundary": t_boundary.cpu(),
        "S_boundary_left": S_boundary_left.cpu(),
        "S_boundary_right": S_boundary_right.cpu(),
        "base_state_dict": base_state_dict,
        "K": K,
        "T": T,
        "S_max": S_max,
        "lambda_data": lambda_data,
        "lambda_pde": lambda_pde,
        "lambda_terminal": lambda_terminal,
        "lambda_boundary": lambda_boundary,
    },
    "notebook11_random_realization.pt"
)

print("Saved: notebook11_random_realization.pt")


In [ ]:
# Train ONE PINN for a given n_data
def train_pinn_for_n_data(
    n_data
):

    start_time = time.time()


    # Use exactly the same observed subset as Regression
    (
        selected_idx,
        t_train,
        S_train,
        V_train,
        u_train
    ) = get_training_subset(
        n_data
    )


    # Fresh model
    #
    # Load identical shared initialization

    model = InversePINN(

        t_min=t_min,
        t_max=t_max,

        S_min=S_min,
        S_max=S_max,

        hidden_width=hidden_width,
        hidden_layers=hidden_layers,

        r_initial=r_initial,
        sigma_initial=sigma_initial

    ).to(
        device=device,
        dtype=dtype
    )


    model.load_state_dict(
        copy.deepcopy(
            base_state_dict
        )
    )


    # Optimizer

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # History

    history = {

        "epoch": [],

        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],

        "r_estimate": [],
        "sigma_estimate": []
    }


    print()
    print(
        "=" * 70
    )

    print(
        f"Training PINN with n_data = {n_data}"
    )

    print(
        "=" * 70
    )


    # Training

    for epoch in range(
        1,
        num_epochs + 1
    ):

        model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        losses = compute_losses(

            model=model,

            t_train=t_train,
            S_train=S_train,
            u_train=u_train,

            t_f=t_f,
            S_f=S_f,

            t_terminal=t_terminal,
            S_terminal=S_terminal,

            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right
        )


        losses[
            "total"
        ].backward()


        optimizer.step()


        # Save history

        if (
            epoch == 1

            or epoch
            % history_every
            == 0

            or epoch
            == num_epochs
        ):

            history[
                "epoch"
            ].append(
                epoch
            )


            history[
                "total_loss"
            ].append(
                losses[
                    "total"
                ].item()
            )


            history[
                "data_loss"
            ].append(
                losses[
                    "data"
                ].item()
            )


            history[
                "pde_loss"
            ].append(
                losses[
                    "pde"
                ].item()
            )


            history[
                "terminal_loss"
            ].append(
                losses[
                    "terminal"
                ].item()
            )


            history[
                "boundary_loss"
            ].append(
                losses[
                    "boundary"
                ].item()
            )


            history[
                "r_estimate"
            ].append(
                model.r.item()
            )


            history[
                "sigma_estimate"
            ].append(
                model.sigma.item()
            )


        # Print progress
        if (
            epoch == 1

            or epoch
            % print_every
            == 0

            or epoch
            == num_epochs
        ):

            print(

                f"Epoch {epoch:5d} | "

                f"Total: "
                f"{losses['total'].item():.6e} | "

                f"Data: "
                f"{losses['data'].item():.6e} | "

                f"PDE: "
                f"{losses['pde'].item():.6e} | "

                f"r: "
                f"{model.r.item():.8f} | "

                f"sigma: "
                f"{model.sigma.item():.8f}"
            )


    # Final PINN parameters

    r_pinn = (
        model.r.item()
    )


    sigma_pinn = (
        model.sigma.item()
    )

    # TEST SET

    model.eval()


    with torch.no_grad():

        # MSE #2
        #
        # NN itself predicts u = V/K
        # Convert back to physical V
        u_pinn_test = model(
            t_test,
            S_test
        )


        V_pinn_nn_test = (
            K
            * u_pinn_test
        )


        mse_2_pinn_nn = (
            torch.mean(

                (
                    V_pinn_nn_test
                    - V_test
                ) ** 2

            )
            .item()
        )


        # MSE #3
        #
        # Take r_PINN and sigma_PINN
        # and plug them into Eq.(5)

        V_pinn_eq5_test = (
            black_scholes_call(

                t=t_test,
                S=S_test,

                K=K,
                T=T,

                r=r_pinn,
                sigma=sigma_pinn
            )
        )


        mse_3_pinn_eq5 = (
            torch.mean(

                (
                    V_pinn_eq5_test
                    - V_test
                ) ** 2

            )
            .item()
        )


    runtime_minutes = (

        time.time()
        - start_time

    ) / 60.0


    # Result dictionary

    result = {

        "n_data":
            n_data,

        "r_pinn":
            r_pinn,

        "sigma_pinn":
            sigma_pinn,

        "mse_2_pinn_nn_test":
            mse_2_pinn_nn,

        "mse_3_pinn_eq5_test":
            mse_3_pinn_eq5,

        "runtime_minutes":
            runtime_minutes
    }


    print()

    print(
        f"n_data = {n_data} completed"
    )

    print()

    print(
        f"r_PINN = "
        f"{r_pinn:.10f}"
    )

    print(
        f"sigma_PINN = "
        f"{sigma_pinn:.10f}"
    )

    print()

    print(
        "MSE #2 PINN NN =",
        mse_2_pinn_nn
    )

    print(
        "MSE #3 PINN r,sigma -> Eq.(5) =",
        mse_3_pinn_eq5
    )

    print()

    print(
        f"Runtime = "
        f"{runtime_minutes:.2f} minutes"
    )

    print(
        "-" * 70
    )


    return (

        result,

        model,

        history,

        V_pinn_nn_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1),

        V_pinn_eq5_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


In [ ]:

# Run BOTH Regression and PINN for every n_data
results = []

# Fixed test-set rows
test_predictions_df = (
    real_df
    .iloc[test_idx_np][
        [
            "original_row",
            "t",
            "S",
            "V"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

for n_data in data_sizes:

    # 1. Regression on exactly sample_indices[n_data]
    (
        regression_result_n,
        V_regression_test_n
    ) = fit_regression_for_n_data(
        n_data
    )

    # 2. PINN on exactly the same sample_indices[n_data]
    #    All n_data PINNs use the same fixed physics points.
    (
        pinn_result_n,
        model_n,
        history_n,
        V_pinn_nn_test_n,
        V_pinn_eq5_test_n
    ) = train_pinn_for_n_data(
        n_data
    )

    # Merge Regression + PINN results for this n_data
    result_n = {
        **regression_result_n,
        **pinn_result_n,
    }

    results.append(
        result_n
    )

    # Save test predictions for this n_data
    test_predictions_df[
        f"regression_eq5_n{n_data}"
    ] = V_regression_test_n

    test_predictions_df[
        f"pinn_nn_n{n_data}"
    ] = V_pinn_nn_test_n

    test_predictions_df[
        f"pinn_eq5_n{n_data}"
    ] = V_pinn_eq5_test_n

    # Save PINN training history
    pd.DataFrame(
        history_n
    ).to_csv(
        f"notebook11_history_n{n_data}.csv",
        index=False
    )

    # Save PINN checkpoint + corresponding Regression result
    torch.save(
        {
            "n_data": n_data,
            "model_state_dict": model_n.state_dict(),
            "selected_indices": sample_indices[n_data].cpu(),
            "train_pool_indices": train_pool_idx.cpu(),
            "test_indices": test_idx.cpu(),
            "regression_result": regression_result_n,
            "pinn_result": pinn_result_n,
            "combined_result": result_n,
            "K": K,
            "T": T,
            "t_min": t_min,
            "t_max": t_max,
            "S_min": S_min,
            "S_max": S_max,
            "S_data_max": S_data_max,
            "N_f": N_f,
            "N_terminal": N_terminal,
            "N_boundary": N_boundary,
            "hidden_width": hidden_width,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "num_epochs": num_epochs,
            "lambda_data": lambda_data,
            "lambda_pde": lambda_pde,
            "lambda_terminal": lambda_terminal,
            "lambda_boundary": lambda_boundary,
            "split_seed": SPLIT_SEED,
            "physics_seed": PHYSICS_SEED,
            "model_seed": MODEL_SEED,
        },
        f"notebook11_pinn_n{n_data}.pt"
    )

    # Save summary after each completed n_data
    pd.DataFrame(
        results
    ).to_csv(
        "notebook11_test_mse_results.csv",
        index=False
    )

    del model_n

# Save all test predictions
test_predictions_df.to_csv(
    "notebook11_test_predictions.csv",
    index=False
)

results_df = pd.DataFrame(
    results
)

print()
print("All n_data experiments completed.")


In [ ]:
final_columns = [
    "n_data",
    "r_tilde",
    "sigma_tilde",
    "regression_train_mse",
    "r_pinn",
    "sigma_pinn",
    "mse_1_regression_eq5_test",
    "mse_2_pinn_nn_test",
    "mse_3_pinn_eq5_test",
    "runtime_minutes",
]

final_results = (
    results_df[
        final_columns
    ]
    .copy()
)

final_results


In [ ]:
results_df[
    [
        "n_data",
        "r_tilde",
        "sigma_tilde",
        "r_pinn",
        "sigma_pinn",
        "mse_1_regression_eq5_test",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test",
    ]
]


In [ ]:
for n_data in data_sizes:

    history = pd.read_csv(
        f"notebook11_history_n{n_data}.csv"
    )

    print(
        "\nn_data =", n_data
    )

    print(
        history.iloc[-1][
            [
                "total_loss",
                "data_loss",
                "pde_loss",
                "terminal_loss",
                "boundary_loss",
                "r_estimate",
                "sigma_estimate"
            ]
        ]
    )


In [ ]:
comparison = results_df[
    [
        "n_data",
        "r_tilde",
        "sigma_tilde",
        "r_pinn",
        "sigma_pinn",
        "mse_1_regression_eq5_test",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test"
    ]
]

comparison
